# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamza44-26/ml-internship-week1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because it produces something a person can act on this week, not just an observation. The starter dataset already shows a large, prioritizable review queue: about half of all pages are trending down while still getting search demand, and pages sitting on page 1 (the most valuable real estate) are *more likely than not* to be declining. That means there's real work here — figuring out which of these pages to look at first, given limited editor hours — and the "which ones first" shape of the question maps directly onto ranking/scoring, which is what this lane is built for.

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("Rows, columns:", df.shape)
print("Clients:", df['client_id'].nunique())
print(df['trend_direction'].value_counts())

Rows, columns: (30000, 44)
Clients: 32
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Given limited hours in a week, which content pages should a content/SEO editor review first — and should the action be refresh, expand, protect, prune, or just monitor?

**Who acts:** A content editor or SEO strategist managing many client pages at once (this is a FlyRank-style agency setting, working across 32 clients here).

**Action:** The editor opens the top item in a ranked queue, reads the reason code (e.g. "page-1 and declining", "stale and still visible"), and decides whether to rewrite/expand the page, protect it as-is, prune it, or leave it on a watch list.

**Cost of a wrong call:**
- **False positive** (flagged as urgent, but it wasn't really at risk): wasted editor hours on a page that didn't need attention — and possibly time taken away from a page that did.
- **False negative** (a page IS losing valuable position and is NOT flagged): the client keeps silently losing page-1 traffic — the most valuable inventory — while nobody looks at it. Given how many page-1 pages are already declining in this data, missing one is the costlier mistake of the two, since page-1 traffic is worth far more per page than lower-tier traffic.

The unit of analysis is **one content page** (one row in the dataset) — that's the level an editor actually acts at.

**Why this isn't just "train a model":** a simple rule already half-exists in this data (`trend_direction == 'down'`), and a rule alone would work if only one signal mattered. But the useful review queue depends on combining several signals at once — how stale a page is, how much demand it still has, what tier it ranks in, how long it's been declining — and weighing them against each other so the highest-value, most-at-risk pages float to the top instead of just the biggest raw drop. That's a scoring/ranking problem, not a single if-statement, which is why this is framed as data/ML work and not just a dashboard filter.

In [2]:
page1 = df[df['position_tier'] == 'page_1']
page1_declining = page1[page1['trend_direction'] == 'down']

print("Page-1 tier pages (the most valuable inventory):", len(page1), 
      f"({len(page1)/len(df):.1%} of all pages)")
print("Of those, currently declining:", len(page1_declining),
      f"({len(page1_declining)/len(page1):.1%} of page-1 pages)")

Page-1 tier pages (the most valuable inventory): 11814 (39.4% of all pages)
Of those, currently declining: 6730 (57.0% of page-1 pages)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the starter dataset (30,000 rows, 32 clients) that support this lane:

In [3]:
declining_with_demand = df[(df['trend_direction'] == 'down') & (df['impressions_last_30d'] > 0)]
print(f"1) Declining pages that still get search demand: {len(declining_with_demand)} "
      f"({len(declining_with_demand)/len(df):.1%} of all pages) — these are live refresh candidates, "
      f"not dead pages.")

stale_visible = df[(df['freshness_tier'].isin(['91-180', '181+'])) &
                    (df['avg_position'] > 0) & (df['avg_position'] <= 20)]
print(f"2) Stale (not updated in 91+ days) AND still visible in the top 20: {len(stale_visible)} "
      f"({len(stale_visible)/len(df):.1%} of all pages) — a clear 'refresh before it fully decays' group.")

print(f"3) Page-1 pages that are currently declining: {len(page1_declining)} "
      f"({len(page1_declining)/len(page1):.1%} of page-1 pages) — the highest-value pages are not safe by default.")

1) Declining pages that still get search demand: 14867 (49.6% of all pages) — these are live refresh candidates, not dead pages.
2) Stale (not updated in 91+ days) AND still visible in the top 20: 6106 (20.4% of all pages) — a clear 'refresh before it fully decays' group.
3) Page-1 pages that are currently declining: 6730 (57.0% of page-1 pages) — the highest-value pages are not safe by default.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- These are **observed, trailing-90-day patterns** in this snapshot of data — not predictions about the future by themselves.
- A ranking/score built from this lane is **decision-support**: it helps an editor prioritize where to look first, given limited time. It is a recommendation, not an instruction.
- Any relationship I find between signals (freshness, position, word count) and decline is **directional** — "pages with X tend to also have Y" — not proof that one causes the other.

**What I can never claim:**
- I cannot claim **causal proof** that refreshing a page will fix its decline — I only observe what happened, I never experiment (no A/B test).
- I cannot claim to be **"predicting Google"** or reverse-engineering the search algorithm — I only see the outcomes (clicks, position, impressions), not the ranking mechanism that produced them.
- `trend_direction` and `trend_pct` are themselves *derived* from past data (they describe what already happened) — they are useful as **labels or evidence**, but I must not treat them as guaranteed future outcomes, and I will never use them as an input feature to "predict" themselves.
- I cannot generalize confidently beyond these 32 clients and this 90-day window to all content, all industries, or all time periods.

In [4]:
# Sanity check: no client names, raw URLs, or private identifiers appear in this notebook —
# only pseudonymized IDs and aggregate counts/percentages, as required by the data-use rules.
print("content_id sample (pseudonymized):", df['content_id'].iloc[0])
print("client_id sample (pseudonymized):", df['client_id'].iloc[0])

content_id sample (pseudonymized): content_304f48230142
client_id sample (pseudonymized): client_f369cb89fc


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.